# Retry: multiplicity issues

Graphs flagged as `mult_issue` in `exact_certs.csv` have at least one irrep appearing with multiplicity $> 1$ in the isotypic decomposition of $\mathcal{E}_\lambda$. This causes `_certify_exact` to bail out since Theorem 7.3 requires multiplicity 1.

We first check **Theorem 6.7**: a rank-1 orbit-isometric embedding is guaranteed to exist when $m_j \leq \dim_D U_j$ for every irrep $U_j$, where $D$ is the associated division algebra. For real type (FS=1): $\dim_D = d_j$; complex type (FS=0): $\dim_D = d_j/2$; quaternionic type (FS=$-1$): $\dim_D = d_j/4$.

Graphs that fail Theorem 6.7 are noted and skipped. For those that pass, we attempt the polyhedral certification anyway — $\mathrm{Cone}\{\ell_\Psi(\varphi_j)\} \subseteq \ell_\Psi(\mathcal{E}_\lambda)$ always holds, so a feasible LP still yields a valid certificate.

In [5]:
import csv, json, os, sys, time

sys.path.insert(0, os.path.abspath('../../cr'))
from orbits import get_edge_orbits
load('../../cr/exact_cert.sage')

pari.allocatemem(1 << 32)  # 4 GB — needed for algebraic arithmetic on larger eigenspaces

DATA_PATH  = os.path.abspath('../raw_hog_data.jsonl')
OUTPUT_CSV = 'exact_certs.csv'
PHI_JSONL  = 'exact_certs.jsonl'

g6_map = {}
with open(DATA_PATH) as f:
    for line in f:
        d = json.loads(line)
        g6_map[d['id']] = d['g6']

with open(OUTPUT_CSV) as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    all_rows = {int(r['id']): r for r in reader}

mult_issue_rows = [
    r for r in all_rows.values()
    if r['lcr_cert'] == 'mult_issue' or r['ucr_cert'] == 'mult_issue'
]
print(f"{len(mult_issue_rows)} mult_issue graphs")

PARI stack size set to 4294967296 bytes, maximum size set to 4294967296
15 mult_issue graphs


## Theorem 6.7 check

Compute the isotypic decomposition for each graph and check $m_j \leq \dim_D U_j$ for all irreps.

In [2]:
def dim_D(d, fs_val):
    if fs_val == 1:  return d        # real type,         D = R
    if fs_val == 0:  return d // 2   # complex type,      D = C
    return d // 4                    # quaternionic type, D = H


def satisfies_thm67(decomp, fs):
    return all(mult <= dim_D(d, fs[idx]) for idx, d, mult in decomp)


# Each entry: (row, direction, which, G, aut, edge_orbits, B, lam, subrep, decomp, fs, ct, classes)
qualifying   = []
disqualified = []

for row in mult_issue_rows:
    hog_id = int(row['id'])
    G = Graph(g6_map[hog_id])
    G.relabel()
    aut     = G.automorphism_group()
    ct      = aut.character_table()
    classes = aut.conjugacy_classes()
    edge_orbits = get_edge_orbits(G, list(aut.gens()))

    for direction, which in [('lcr', 'lambda2'), ('ucr', 'lambdan')]:
        if row[f'{direction}_cert'] != 'mult_issue':
            continue

        lam, B = eigenspace_exact(G, which=which)
        subrep = build_subrep(B, aut)
        chi    = subrep_character(subrep, aut)
        decomp = irrep_decomposition(chi, aut, ct, classes)
        fs     = frobenius_schur_indicators(aut, ct, classes, decomp)

        ok = satisfies_thm67(decomp, fs)
        print(f"HoG {hog_id:5d} ({direction}): {'✓ Thm 6.7' if ok else '✗ Thm 6.7'}  "
              f"decomp={[(i, d, m) for i, d, m in decomp]}")

        if ok:
            qualifying.append((row, direction, which, G, aut, edge_orbits, B, lam, subrep, decomp, fs, ct, classes))
        else:
            disqualified.append((hog_id, direction, decomp, fs))

print(f"\n{len(qualifying)} qualifying  |  {len(disqualified)} disqualified")

HoG   730 (ucr): ✗ Thm 6.7  decomp=[(0, 1, 2)]
HoG 21234 (ucr): ✗ Thm 6.7  decomp=[(0, 1, 2)]
HoG 21609 (ucr): ✗ Thm 6.7  decomp=[(0, 1, 2), (6, 1, 1)]
HoG 50485 (ucr): ✗ Thm 6.7  decomp=[(0, 1, 4), (1, 1, 2)]
HoG  1290 (lcr): ✗ Thm 6.7  decomp=[(0, 1, 2), (1, 1, 1), (3, 1, 1), (6, 2, 1), (7, 2, 1)]
HoG 51486 (ucr): ✓ Thm 6.7  decomp=[(0, 1, 1), (1, 1, 1), (4, 2, 1), (7, 3, 2), (8, 3, 1)]
HoG 50489 (ucr): ✓ Thm 6.7  decomp=[(0, 1, 1), (5, 2, 2), (8, 3, 1), (9, 3, 2)]
HoG 51450 (ucr): ✓ Thm 6.7  decomp=[(0, 1, 1), (1, 1, 1), (5, 2, 2), (6, 3, 1), (7, 3, 2), (8, 3, 2), (9, 3, 1)]
HoG 51453 (ucr): ✗ Thm 6.7  decomp=[(0, 1, 2), (1, 1, 1), (2, 2, 3), (3, 3, 4), (4, 3, 5)]
HoG 51393 (ucr): ✗ Thm 6.7  decomp=[(0, 1, 2), (1, 1, 1), (5, 2, 2), (7, 3, 2), (8, 3, 3), (9, 3, 1)]
HoG 51484 (ucr): ✓ Thm 6.7  decomp=[(0, 1, 1), (4, 3, 1), (5, 3, 1), (6, 4, 1), (7, 4, 1), (8, 5, 2), (9, 5, 1)]
HoG 51485 (ucr): ✓ Thm 6.7  decomp=[(0, 1, 1), (4, 3, 1), (5, 3, 1), (6, 4, 1), (7, 4, 1), (8, 5, 2), (9, 5, 

## Disqualified graphs

The following graphs have $m_j > \dim_D U_j$ for some irrep and do not satisfy Theorem 6.7. A rank-1 orbit-isometric embedding is not guaranteed, and the polyhedral cone approach cannot certify them.

In [3]:
for hog_id, direction, decomp, fs in disqualified:
    print(f"HoG {hog_id} ({direction}):")
    for idx, d, mult in decomp:
        dD   = dim_D(d, fs[idx])
        flag = '  ← violates Thm 6.7' if mult > dD else ''
        print(f"  irrep {idx}: dim={d}, mult={mult}, dim_D={dD}{flag}")

HoG 730 (ucr):
  irrep 0: dim=1, mult=2, dim_D=1  ← violates Thm 6.7
HoG 21234 (ucr):
  irrep 0: dim=1, mult=2, dim_D=1  ← violates Thm 6.7
HoG 21609 (ucr):
  irrep 0: dim=1, mult=2, dim_D=1  ← violates Thm 6.7
  irrep 6: dim=1, mult=1, dim_D=1
HoG 50485 (ucr):
  irrep 0: dim=1, mult=4, dim_D=1  ← violates Thm 6.7
  irrep 1: dim=1, mult=2, dim_D=1  ← violates Thm 6.7
HoG 1290 (lcr):
  irrep 0: dim=1, mult=2, dim_D=1  ← violates Thm 6.7
  irrep 1: dim=1, mult=1, dim_D=1
  irrep 3: dim=1, mult=1, dim_D=1
  irrep 6: dim=2, mult=1, dim_D=2
  irrep 7: dim=2, mult=1, dim_D=2
HoG 51453 (ucr):
  irrep 0: dim=1, mult=2, dim_D=1  ← violates Thm 6.7
  irrep 1: dim=1, mult=1, dim_D=1
  irrep 2: dim=2, mult=3, dim_D=2  ← violates Thm 6.7
  irrep 3: dim=3, mult=4, dim_D=3  ← violates Thm 6.7
  irrep 4: dim=3, mult=5, dim_D=3  ← violates Thm 6.7
HoG 51393 (ucr):
  irrep 0: dim=1, mult=2, dim_D=1  ← violates Thm 6.7
  irrep 1: dim=1, mult=1, dim_D=1
  irrep 5: dim=2, mult=2, dim_D=2
  irrep 7: dim=3, 

## Polyhedral certification for qualifying graphs

Run the isotypic projection pipeline directly, bypassing the multiplicity-1 guard in `_certify_exact`.

In [ ]:
results = {}  # (hog_id, direction) -> (phi_vectors, weights, lam) or reason string

# verify_certificate is slow for graphs with large eigenspaces — LP feasibility
# is sufficient to guarantee a valid certificate, so skip verification here.
DO_VERIFY = False

for row, direction, which, G, aut, edge_orbits, B, lam, subrep, decomp, fs, ct, classes in qualifying:
    hog_id = int(row['id'])
    print(f"\nHoG {hog_id} ({direction})  dim={B.nrows()}  "
          f"decomp={[(i, d, m) for i, d, m in decomp]}")

    t0 = time.time()
    try:
        paired      = pair_complex_conjugates(decomp, ct, fs)
        projectors  = isotypic_projectors(subrep, aut, ct, classes, paired)
        phi_vectors = get_isotypic_representative(projectors, B)
        energies    = orbit_energies(phi_vectors, edge_orbits)
        weights     = exact_weights(energies, edge_orbits)

        if weights is None:
            print(f"  LP infeasible  ({time.time()-t0:.1f}s)")
            results[(hog_id, direction)] = 'infeasible'
        else:
            print(f"  weights={dict(weights)}  ({time.time()-t0:.1f}s)")
            if DO_VERIFY:
                phi = combine_certificate(phi_vectors, weights)
                ok  = verify_certificate(phi, G, lam, edge_orbits)
                print(f"  verify={ok}")
                results[(hog_id, direction)] = (phi_vectors, weights, lam) if ok else 'verify_failed'
            else:
                results[(hog_id, direction)] = (phi_vectors, weights, lam)

    except Exception as e:
        print(f"  ERROR: {e}")
        results[(hog_id, direction)] = f'error: {e}'

print("\nSummary:")
for (hid, direction), val in results.items():
    status = 'exact' if isinstance(val, tuple) else val
    print(f"  HoG {hid:5d} ({direction}): {status}")

## Update CSV and JSONL

In [ ]:
with open(PHI_JSONL, 'a') as f_phi:
    for (hog_id, direction), val in results.items():
        row = all_rows[hog_id]
        cert_col = f'{direction}_cert'
        if isinstance(val, tuple):
            phi_vectors, weights, lam = val
            row[cert_col] = 'exact'
            f_phi.write(json.dumps({
                'id':  hog_id,
                'lcr': serialize_certificate(phi_vectors, weights, lam) if direction == 'lcr' else None,
                'ucr': serialize_certificate(phi_vectors, weights, lam) if direction == 'ucr' else None,
            }) + '\n')
        elif val == 'infeasible':
            row[cert_col] = 'infeasible'
        elif isinstance(val, str) and val.startswith('error'):
            row[cert_col] = 'error'
        # verify_failed and no_cert leave the row as mult_issue

with open(OUTPUT_CSV, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_rows.values())

print(f"Updated {OUTPUT_CSV} and appended to {PHI_JSONL}.")